In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
datasetA=pd.read_csv(r"D:\PRJ1\PROJECT1\data\processed\Acleaned.csv")
datasetR=pd.read_csv(r"D:\PRJ1\PROJECT1\data\processed\Rcleaned.csv")
dataset_AR=datasetR.merge(datasetA[["anime_id","name"]],on="anime_id")

In [7]:
datasetR.head()

,user_id,anime_id,rating
0,3,20,8
1,3,154,6
2,3,170,9
3,3,199,10
4,3,225,9


In [21]:
datasetA.head()

,anime_id,name,genre,type,episodes,rating,members,genre_type_combined
0,32281,kimi no na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,"drama, romance, school, supernatural movie"
1,5114,fullmetal alchemist: brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665,"action, adventure, drama, fantasy, magic, mili..."
2,28977,gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262,"action, comedy, historical, parody, samurai, s..."
3,9253,steins;gate,"Sci-Fi, Thriller",TV,24,9.17,673572,"sci-fi, thriller tv"
4,9969,gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266,"action, comedy, historical, parody, samurai, s..."


In [25]:
dataset_AR[dataset_AR["name"]=="Kimi No Na Wa."]

,user_id,anime_id,rating,name


In [17]:
datasetA.shape

(9875, 8)

In [18]:
dataset_AR.shape

(5695966, 4)

In [20]:
dataset_AR["name"].isin(datasetA).value_counts()

name
False    5695966
Name: count, dtype: int64

In [4]:
datasetA.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

FOR OUR FISRT VERSION LETS BUILD ONLY USING ANIME DATASET AND ONLY REQUIRED ATTRIBUTES FROM THE DATASET(ANIMEID,NAME,GENRE,TYPE)

In [5]:
datasetA=datasetA[["anime_id","name","genre","type"]]
datasetA.head()

,anime_id,name,genre,type
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV


In [9]:
datasetA['genre_type_combined']=(datasetA['genre']+ " " +datasetA['type'])

In [11]:
datasetA.head()

,anime_id,name,genre,type,episodes,rating,members,genre_type_combined
0,32281,kimi no na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,"drama, romance, school, supernatural movie"
1,5114,fullmetal alchemist: brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665,"action, adventure, drama, fantasy, magic, mili..."
2,28977,gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262,"action, comedy, historical, parody, samurai, s..."
3,9253,steins;gate,"Sci-Fi, Thriller",TV,24,9.17,673572,"sci-fi, thriller tv"
4,9969,gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266,"action, comedy, historical, parody, samurai, s..."


In [10]:
datasetA['genre_type_combined']=datasetA['genre_type_combined'].str.lower()
datasetA['name']=datasetA['name'].str.lower()


TF_IDF

In [9]:
tf_idf=TfidfVectorizer(stop_words="english")
tf_idf_matrix=tf_idf.fit_transform(datasetA['genre_type_combined'])

In [10]:
similarity_matrix=cosine_similarity(tf_idf_matrix)
similarity_matrix

array([[1.        , 0.13624096, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.13624096, 1.        , 0.21757873, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.21757873, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 1.        ,
        1.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 1.        ,
        1.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 1.        ,
        1.        ]], shape=(9875, 9875))

In [11]:
ind=pd.Series(datasetA.index,index=datasetA['name']).drop_duplicates()

In [12]:
ind['kimi no na wa.']

np.int64(0)

In [13]:
datasetA['name'][0]

'kimi no na wa.'

In [21]:
datasetA[datasetA['name']=="one punch man"]

,anime_id,name,genre,type,genre_type_combined
23,30276,one punch man,"Action, Comedy, Parody, Sci-Fi, Seinen, Super ...",TV,"action, comedy, parody, sci-fi, seinen, super ..."


**FIRST ML LOGIC (CONTENT BASED):**

In [14]:
def recomm_ani(ani_title,no_recomm=50):
    indices=ind[ani_title]
    sim_score=list(enumerate(similarity_matrix[indices]))
    sim_score=sorted(sim_score,key=lambda x: x[1],reverse=True)
    sim_score=sim_score[1:no_recomm+1]
    recomm=[]
    for ani_index,score in sim_score:
        recomm.append({"Name":datasetA.iloc[ani_index]["name"],"score":float(score)})
    return pd.DataFrame(recomm)

In [15]:
recomm_ani("death note")

,Name,score
0,mousou dairinin,0.967700
1,death note rewrite,0.943175
2,higurashi no naku koro ni kai,0.882864
3,mirai nikki (tv),0.822645
4,higurashi no naku koro ni rei,0.816495
5,higurashi no naku koro ni,0.800059
6,monster,0.796635
7,mirai nikki (tv): ura mirai nikki,0.751624
8,zankyou no terror,0.732045
9,shigofumi,0.725725


In [24]:
datasetA[datasetA['name']=="mousou dairinin"]

,anime_id,name,genre,type,genre_type_combined
961,323,mousou dairinin,"Drama, Mystery, Police, Psychological, Superna...",TV,"drama, mystery, police, psychological, superna..."


In [16]:
import joblib
joblib.dump(tf_idf,r"D:\PRJ1\PROJECT1\models\tf_idf.pkl")
joblib.dump(tf_idf_matrix,r"D:\PRJ1\PROJECT1\models\tf_idf_matrix.pkl")
joblib.dump(ind,r"D:\PRJ1\PROJECT1\models\ind.pkl")
joblib.dump(similarity_matrix,r"D:\PRJ1\PROJECT1\models\similarity_matrix.pkl")

['D:\\PRJ1\\PROJECT1\\models\\similarity_matrix.pkl']

In [18]:
content_anime_index={idx:anime for idx,anime in enumerate(datasetA["name"])}
joblib.dump(content_anime_index,r"D:\PRJ1\PROJECT1\models\content_anime_index.pkl")

['D:\\PRJ1\\PROJECT1\\models\\content_anime_index.pkl']